In [ ]:
pip install xlsxwriter

In [ ]:
import pandas as pd
from datetime import datetime
import re

In [ ]:
df=pd.read_excel("/content/Merged_All_Surveys_2026-08-11.xlsx")

In [ ]:
df[["Slot No And Name_2","Slot Name"]]=df["Select Slot No And Name"].str.split('_',expand=True)

In [ ]:
df["District"] = df["ID"].str.extract(r'^([A-Za-z\s]+)')[0].str.strip()

In [ ]:
df[["ID","District"]]

,ID,District
0,Azamgarh000363,Azamgarh
1,Azamgarh000362,Azamgarh
2,Azamgarh000361,Azamgarh
3,Azamgarh000360,Azamgarh
4,Azamgarh000359,Azamgarh
...,...,...
2272,Siddharthnagar000005,Siddharthnagar
2273,Siddharthnagar000004,Siddharthnagar
2274,Siddharthnagar000003,Siddharthnagar
2275,Siddharthnagar000002,Siddharthnagar


In [ ]:

dummy_emails = [
    "test@gmail.com"
    ]

df2 = df[~df["User"].isin(dummy_emails)]

In [ ]:
count=df2.groupby("District")["Select AC NAME"].value_counts().reset_index()
count.head(1)

,District,Select AC NAME,count
0,Azamgarh,Lalganj (351),175


In [ ]:
df3=df2[["District","Select AC NAME","User","Created On"]]

In [ ]:
df23=df2.copy()
df23["Date"] = pd.to_datetime(df23["Created On"])
df23["Date"]=pd.to_datetime(df23["Date"]).dt.date
today = datetime.now().date()

df_ALL=df2[df23["Date"]==today]

No_Candidate=df_ALL[df_ALL["Select Candidate Name"].isnull()]
No_Candidate2=No_Candidate[["District","Select AC NAME","Slot No And Name_2","User"]]
No_Candidate2["Merge"]=No_Candidate2["Select AC NAME"]+"_"+ No_Candidate2["Slot No And Name_2"]
No_Candidate_Count=No_Candidate2.groupby(["District","Select AC NAME","Slot No And Name_2","Merge"])["User"].value_counts().reset_index()
No_Candidate_Count

Today User Count

In [ ]:
df3["Date"] = pd.to_datetime(df3["Created On"])
df3["Date"]=pd.to_datetime(df3["Date"]).dt.date
today = datetime.now().date()
df_today = df3[df3["Date"] == today]

df4 = df_today.groupby(["District", "Select AC NAME","Date"])["User"].nunique().reset_index(name="Unique_User_Count")
df4.head(1)

/tmp/ipykernel_940/315000041.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3["Date"] = pd.to_datetime(df3["Created On"])
/tmp/ipykernel_940/315000041.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3["Date"]=pd.to_datetime(df3["Date"]).dt.date


,District,Select AC NAME,Date,Unique_User_Count
0,Azamgarh,Atrauliya (343),2026-08-11,4


In [ ]:
df5=df_today.groupby("District")["Select AC NAME"].value_counts().reset_index()
df5.head(2)

,District,Select AC NAME,count
0,Azamgarh,Lalganj (351),102
1,Azamgarh,Mubarakpur (346),66


In [ ]:
UC=df4.copy()
UC["Today Sample Count"]=UC["Select AC NAME"].map(df5.set_index("Select AC NAME")["count"])
UC.head(2)

,District,Select AC NAME,Date,Unique_User_Count,Today Sample Count
0,Azamgarh,Atrauliya (343),2026-08-11,4,38
1,Azamgarh,Azamgarh (347),2026-08-11,1,1


Not choosing Slot No

In [ ]:
df6=df_today[df["Select Slot No And Name"].isnull()]
df7=df6.groupby(["District","Select AC NAME"])["User"].value_counts().reset_index()
df7

Party Percentage

In [ ]:
Party=(df2.groupby(["District","Select AC NAME"])["6.C. मतदाता की पसंद:- आगामी 2027 विधानसभा चुनाव में आप किस राजनीतिक दल को वोट देना पसंद करेंगे?"].value_counts(normalize=True)*100).round(2).unstack().reset_index()
Party=Party.fillna(0)
Party["Grand Total"]=Party["Select AC NAME"].map(count.set_index("Select AC NAME")["count"])
Party

Candidate_Percentage

In [ ]:
Candidate_count=df2.groupby(["District","Select AC NAME"])["Select Candidate Name"].value_counts().reset_index()
Candidate_count

In [ ]:
Candidate=(df2.groupby(["District","Select AC NAME"])["Select Candidate Name"].value_counts(normalize=True)*100).round(2).reset_index()
Candidate.rename(columns={"proportion":"Proportion in Assembly"},inplace=True)
# Candidate["Candidate Count"]=Candidate["Select Candidate Name"].map(Candidate_count.set_index("Select Candidate Name")["count"])
Candidate

In [ ]:
df2["New_Village"]=df2["Village"].str.lower()
df2["New_Village"]=df2["Village"].str.capitalize()

/tmp/ipykernel_940/3179651593.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2["New_Village"]=df2["Village"].str.lower()
/tmp/ipykernel_940/3179651593.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2["New_Village"]=df2["Village"].str.capitalize()


Village Checking

In [ ]:
def get_village_names(slot):
    # Extract the part before the parentheses
    village_part = re.split(r'\s*\(', str(slot))[0]
    # Split by comma and strip spaces
    villages = [v.strip() for v in village_part.split(',')]
    return villages

# Function to check if first 3 characters of village exist in slot
def check_village_in_slot(slot, village):
    village_names = get_village_names(slot)
    # Get first 3 characters of the village (case-insensitive)
    village_prefix = village[:3].lower()
    # Check if any village in slot starts with these 3 characters
    return any(v.lower().startswith(village_prefix) for v in village_names)

# Apply to your DataFrame
df2['Village_Exists'] = df2.apply(
    lambda row: check_village_in_slot(row['Slot Name'], row['New_Village']),
    axis=1
)

/tmp/ipykernel_940/3518647366.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['Village_Exists'] = df2.apply(


Caste wise Party


In [ ]:
Caste=df2[["District","Select AC NAME","Slot No And Name_2","Caste","6.C. मतदाता की पसंद:- आगामी 2027 विधानसभा चुनाव में आप किस राजनीतिक दल को वोट देना पसंद करेंगे?"]]
Caste["Merge"]=Caste["Select AC NAME"]+"_"+ Caste["Slot No And Name_2"]+"_"+Caste["Caste"]
Caste_Count=Caste.groupby("Merge")["Caste"].value_counts().reset_index()
# Caste_Count["MERGE_ALL"]=Caste_Count["Merge"]+"_"+Caste_Count["Caste"]
# Caste_All=Caste_Count.pop("MERGE_ALL")
# Caste_Count.insert(2,"MERGE_ALL",Caste_All)
Caste_Count


In [ ]:

Caste2=(Caste.groupby(["District","Merge","Caste"])["6.C. मतदाता की पसंद:- आगामी 2027 विधानसभा चुनाव में आप किस राजनीतिक दल को वोट देना पसंद करेंगे?"].value_counts(normalize=True)*100).round(2).unstack().reset_index()

Caste2["Grand Total"]=Caste2["Merge"].map(Caste_Count.set_index("Merge")["count"])
Caste2=Caste2.fillna(0)
Caste2

Slot me kitne sample hue hai

In [ ]:
Sample=df2[["District","Select AC NAME","Slot No And Name_2",]]
Sample2=Sample.value_counts().reset_index()

To Correct the village name

In [ ]:


def extract_correct_village_name(slot_name, new_village):
    """
    Extract correct village name by matching first 3 characters
    from Slot Name with New_Village
    """
    # Handle NaN or empty values
    if pd.isna(slot_name) or pd.isna(new_village):
        return new_village if not pd.isna(new_village) else ""

    # Convert to string
    slot_name = str(slot_name)
    new_village = str(new_village)

    # Remove parentheses and their contents from Slot Name
    cleaned = re.sub(r'\([^)]*\)', '', slot_name)

    # Split by comma and get all parts
    parts = [p.strip() for p in cleaned.split(',') if p.strip()]

    # Get first 3 characters of New_Village for matching
    new_village_first3 = new_village[:3].lower()

    # Find the part that matches first 3 characters
    for part in parts:
        if part[:3].lower() == new_village_first3:
            return part  # Return the correctly spelled name from Slot Name

    # If no match found, return the last part as fallback
    return parts[-1] if parts else new_village

# Apply the function to create Modified Village column
df2['Modified Village'] = df2.apply(
    lambda row: extract_correct_village_name(row['Slot Name'], row['New_Village']),
    axis=1
)


/tmp/ipykernel_940/2353611096.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['Modified Village'] = df2.apply(


In [ ]:
with pd.ExcelWriter('Error_UP_Report.xlsx', engine='xlsxwriter') as writer:
    df2.to_excel(writer, sheet_name="All Data",index=False)
    count.to_excel(writer, sheet_name="Overall_Count",index=False)
    UC.to_excel(writer, sheet_name='Today_User_Count', index=False)
    df7.to_excel(writer, sheet_name='Slot_Blank', index=False)
    No_Candidate_Count.to_excel(writer, sheet_name="No_Candidate", index=False)
    Party.to_excel(writer, sheet_name='Party_Percentage', index=False)
    Candidate_count.to_excel(writer, sheet_name='Candidate_count', index=False)
    Candidate.to_excel(writer, sheet_name='Candidate_Percentage', index=False)
    Caste2.to_excel(writer, sheet_name="Caste VS Party", index=False)
    Sample2.to_excel(writer, sheet_name="Sample_Count", index=False)

